# Entrega 3: Preprocesamiento Estructural, Modelo y Métricas

**Proyecto:** Dinámica del comercio mundial: exportaciones e importaciones por país y región geográfica (1989-2023)

**Contexto:** En el Entregable 2, el dataset fue perfilado y limpiado exhaustivamente. El objetivo de este notebook es ejecutar el **preprocesamiento estructural** y evaluar **tres opciones arquitectónicas** (Tabla Plana como alternativa base, Modelo Relacional 3NF y Esquema en Estrella) mediante benchmarking para seleccionar el modelo final para Tableau.

In [13]:
import pandas as pd
import numpy as np
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos Limpios y Alternativa Base (Tabla Plana)

In [14]:
df_base = pd.read_csv('../data/processed/dataset_limpio_entrega2_consolidado.csv')


print(f'Alternativa Base (Tabla Plana) cargada: {df_base.shape[0]} filas.')
df_obt = df_base.copy()

Alternativa Base (Tabla Plana) cargada: 7783 filas.


## 2. Preprocesamiento Estructural: Construcción de los Modelos Alternativos

### Contexto
La limpieza de datos (nulos, atípicos, tipado, filtrado de entidades) fue resuelta exhaustivamente en la **Entrega 2** y está documentada en `docs/entregable2_documentacion_detallada.md`. El preprocesamiento de esta entrega es **estrictamente estructural/arquitectónico**: su objetivo es transformar la matriz plana en modelos relacionales para evaluar su idoneidad en Tableau.

### Modelos Evaluados

**Alternativa Base — Tabla Plana:** El dataset tal como sale de la Entrega 2, sin transformación estructural. Sirve como línea base para cuantificar la degradación analítica de no modelar.

**Opción 1 — Modelo Relacional 3NF (Inmon):** Normalización en Tercera Forma Normal usando **claves naturales** (strings). Las tablas se derivan de las dependencias funcionales sin introducir surrogate keys artificiales:
- `Dim_Partners_3NF(partner_name)` — clave natural string
- `Dim_Macro_3NF(year, world_growth)` — clave natural integer
- `Fact_Trade_3NF(partner_name FK, year FK, export)` — FK textual sobre `partner_name`

**Opción 2 — Esquema en Estrella (Kimball):** Modelo dimensional con **surrogate keys enteras**. Resuelve las mismas dependencias funcionales que 3NF pero reemplaza las claves naturales por enteros incrementales:
- `Dim_Country(dim_country_sk, partner_name)` — surrogate key int64
- `Dim_Time(dim_time_sk, year, world_growth)` — surrogate key int64
- `Fact_Trade(dim_country_sk FK, dim_time_sk FK, export)` — FK exclusivamente int64

### Pasos de Transformación (comunes a ambas opciones)

**Paso 1 — Validación de unicidad:** Se verifica que `(Partner Name, Year)` sea clave primaria única. Sin esta garantía, cualquier join produce un producto cartesiano que infla las métricas.

**Paso 2 — Normalización 2NF:** `World Growth (%)` depende únicamente del año, no del país. Almacenarla en cada fila transaccional viola la 2NF y produce fan-out trap al agregar. Se extrae a `Dim_Macro_3NF` / `Dim_Time`.

**Paso 3 — Tipo de clave (diferencia central entre modelos):** 3NF mantiene `Partner Name` (string) como FK en la tabla de hechos; Estrella lo reemplaza por `dim_country_sk` (int64). Esta decisión es la fuente principal de divergencia en las métricas 1 y 4.

**Paso 4 — Tabla de Hechos angosta:** En ambos modelos el fact queda reducido a 3 columnas (2 FKs + 1 métrica aditiva), eliminando toda redundancia descriptiva.

In [15]:
# Paso 1 — Validación de unicidad
pk_unique = df_base.set_index(['Partner Name', 'Year']).index.is_unique
print(f'Clave primaria (Partner Name, Year) única: {pk_unique}')

# ──────────────────────────────────────────────────────────────
# OPCIÓN 1: Modelo Relacional 3NF (Inmon) — claves naturales
# Partner Name permanece como FK string en la tabla de hechos
# ──────────────────────────────────────────────────────────────
dim_partners_3nf = df_base[['Partner Name']].drop_duplicates().reset_index(drop=True)
dim_macro_3nf    = df_base[['Year', 'World Growth (%)']].drop_duplicates().reset_index(drop=True)
fact_3nf         = df_base[['Partner Name', 'Year', 'Export (US$ Million)']].copy()

print(f'\nModelo 3NF (Inmon) — claves naturales:')
print(f'  Dim_Partners_3NF : {dim_partners_3nf.shape[0]} filas | {dim_partners_3nf.shape[1]} col  | clave: Partner Name (string)')
print(f'  Dim_Macro_3NF    : {dim_macro_3nf.shape[0]} filas | {dim_macro_3nf.shape[1]} cols | clave: Year (int)')
print(f'  Fact_Trade_3NF   : {fact_3nf.shape[0]} filas | {fact_3nf.shape[1]} cols | FK: Partner Name (string)')

# ──────────────────────────────────────────────────────────────
# OPCIÓN 2: Esquema en Estrella (Kimball) — surrogate keys
# Las FK en fact_trade son int64; el string queda solo en dim
# ──────────────────────────────────────────────────────────────
dim_country = df_base[['Partner Name']].drop_duplicates().reset_index(drop=True)
dim_country.insert(0, 'dim_country_sk', range(1, 1 + len(dim_country)))

dim_time = df_base[['Year', 'World Growth (%)']].drop_duplicates().reset_index(drop=True)
dim_time.insert(0, 'dim_time_sk', range(1, 1 + len(dim_time)))

fact_trade = df_base.merge(dim_country[['Partner Name', 'dim_country_sk']], on='Partner Name', how='left')
fact_trade = fact_trade.merge(dim_time[['Year', 'dim_time_sk']], on='Year', how='left')
fact_trade = fact_trade[['dim_time_sk', 'dim_country_sk', 'Export (US$ Million)']]

print(f'\nEsquema en Estrella (Kimball) — surrogate keys:')
print(f'  Dim_Country : {dim_country.shape[0]} filas | {dim_country.shape[1]} cols | clave: dim_country_sk (int64)')
print(f'  Dim_Time    : {dim_time.shape[0]} filas | {dim_time.shape[1]} cols | clave: dim_time_sk (int64)')
print(f'  Fact_Trade  : {fact_trade.shape[0]} filas | {fact_trade.shape[1]} cols | FK: int64 exclusivamente')

Clave primaria (Partner Name, Year) única: True

Modelo 3NF (Inmon) — claves naturales:
  Dim_Partners_3NF : 252 filas | 1 col  | clave: Partner Name (string)
  Dim_Macro_3NF    : 34 filas | 2 cols | clave: Year (int)
  Fact_Trade_3NF   : 7783 filas | 3 cols | FK: Partner Name (string)

Esquema en Estrella (Kimball) — surrogate keys:
  Dim_Country : 252 filas | 2 cols | clave: dim_country_sk (int64)
  Dim_Time    : 34 filas | 3 cols | clave: dim_time_sk (int64)
  Fact_Trade  : 7783 filas | 3 cols | FK: int64 exclusivamente


## 3. Pruebas de Benchmarking (Evaluación de Modelos)
Métricas empíricas para comparar los tres modelos: **Tabla Plana (Base)**, **3NF Inmon (Opción 1)** y **Esquema en Estrella (Opción 2)**.

In [16]:
resultados_bench = {}

# Métrica 1 — Huella de Memoria RAM Total (KB)
mem_plana = df_obt.memory_usage(deep=True).sum() / 1024
mem_3nf   = (dim_partners_3nf.memory_usage(deep=True).sum() +
             dim_macro_3nf.memory_usage(deep=True).sum() +
             fact_3nf.memory_usage(deep=True).sum()) / 1024
mem_star  = (dim_country.memory_usage(deep=True).sum() +
             dim_time.memory_usage(deep=True).sum() +
             fact_trade.memory_usage(deep=True).sum()) / 1024

resultados_bench['Memoria RAM (KB)'] = {
    'Tabla Plana (Base)':    round(mem_plana, 2),
    '3NF Inmon (Opción 1)': round(mem_3nf,   2),
    'Estrella (Opción 2)':  round(mem_star,  2),
}

# Métrica 2 — Integridad de Agregación Macro (Fan-Out Trap)
# World Growth (%) se repite N veces por año en la Tabla Plana → promedio distorsionado
avg_plana = df_obt['World Growth (%)'].mean()
avg_3nf   = dim_macro_3nf['World Growth (%)'].mean()
avg_star  = dim_time['World Growth (%)'].mean()

resultados_bench['Integridad Macro (Avg Growth %)'] = {
    'Tabla Plana (Base)':    f'{avg_plana:.2f}% (distorsionado)',
    '3NF Inmon (Opción 1)': f'{avg_3nf:.2f}% (real)',
    'Estrella (Opción 2)':  f'{avg_star:.2f}% (real)',
}

# Métrica 3 — Anomalía de Actualización (World Growth corregido por el Banco Mundial)
# ¿Cuántas filas deben actualizarse si se corrige el dato de un año?
filas_plana_2010 = df_obt[df_obt['Year'] == 2010].shape[0]
filas_3nf_2010   = dim_macro_3nf[dim_macro_3nf['Year'] == 2010].shape[0]
filas_star_2010  = dim_time[dim_time['Year'] == 2010].shape[0]

resultados_bench['Anomalía de Actualización (filas afectadas)'] = {
    'Tabla Plana (Base)':    f'{filas_plana_2010} filas (una por país activo en 2010)',
    '3NF Inmon (Opción 1)': f'{filas_3nf_2010} fila (solo en Dim_Macro_3NF)',
    'Estrella (Opción 2)':  f'{filas_star_2010} fila (solo en Dim_Time)',
}

# Métrica 4 — Anomalía de Nombre (cambio en el nombre oficial de un país)
# ¿Cuántas filas deben actualizarse si un país cambia su nombre oficial?
# En 3NF la FK del fact es el string → hay que actualizar cada fila del fact que lo referencie
# En Estrella la FK del fact es int64 → el fact queda intacto; solo cambia 1 fila en Dim_Country
test_pais     = df_obt['Partner Name'].value_counts().idxmax()
rows_plana_nk = df_obt[df_obt['Partner Name'] == test_pais].shape[0]
rows_3nf_nk   = fact_3nf[fact_3nf['Partner Name'] == test_pais].shape[0] + 1  # fact + dim
rows_star_nk  = dim_country[dim_country['Partner Name'] == test_pais].shape[0]  # solo dim; fact intacto

resultados_bench['Anomalía de Nombre (filas afectadas)'] = {
    'Tabla Plana (Base)':    f'{rows_plana_nk} filas (en dataset completo)',
    '3NF Inmon (Opción 1)': f'{rows_3nf_nk} filas (fact + dim; FK es string)',
    'Estrella (Opción 2)':  f'{rows_star_nk} fila (Dim_Country; fact intacto con SK int)',
}

print('Métricas extraídas exitosamente.')
print(f'  País de prueba (métrica 4): {test_pais}')

Métricas extraídas exitosamente.
  País de prueba (métrica 4): Afghanistan


## 4. Tabla Comparativa Formal y Decisión
Se justifica la elección final en base a los datos empíricos.

In [19]:
df_comparativo = pd.DataFrame(resultados_bench).T
df_comparativo.index.name = 'Métrica Analítica'
df_comparativo.reset_index(inplace=True)

df_comparativo['Conclusión de Evaluación'] = [
    ("Tabla Plana almacena todas las columnas descriptivas en cada fila transaccional. "
     "3NF normaliza las dimensiones pero mantiene 'Partner Name' como FK string en el "
     "fact (7 783 strings repetidos). Estrella reemplaza ese string por int64 → "
     "mínima huella en la tabla de hechos."),
    ("Tabla Plana repite World Growth % una vez por país/año: su promedio queda "
     "ponderado por el N de países activos cada año → fan-out trap. 3NF y Estrella "
     "aíslan el indicador en una tabla anual (Dim_Macro / Dim_Time): un único valor "
     "por año garantiza el promedio real."),
    ("3NF y Estrella aíslan World Growth en tablas dedicadas: corregir un año "
     "requiere 1 fila. En Tabla Plana la corrección se propaga a todas las filas "
     "del año (una por país activo en 2010)."),
    ("En Tabla Plana y 3NF el 'Partner Name' es texto en el fact: un cambio de nombre "
     "oficial exige actualizar cada fila que lo referencie. Paradójicamente, 3NF "
     "resulta levemente peor que Tabla Plana al requerir las mismas filas del fact "
     "más 1 fila adicional en Dim_Partners_3NF. Estrella es inmune: el fact solo "
     "almacena el int64 surrogate; basta actualizar 1 fila en Dim_Country."),
]

VERDE    = 'background-color: #c6efce; color: #276221; font-weight: bold'
ROJO     = 'background-color: #ffc7ce; color: #9c0006'

# Métricas 2 y 3: tanto 3NF como Estrella son correctos → ambos son ganadores
# Métrica 4: tanto Tabla Plana como 3NF fallan → ambos son perdedores
GANADORES = {
    'Memoria RAM (KB)':                            ['Estrella (Opción 2)'],
    'Integridad Macro (Avg Growth %)':             ['3NF Inmon (Opción 1)', 'Estrella (Opción 2)'],
    'Anomalía de Actualización (filas afectadas)': ['3NF Inmon (Opción 1)', 'Estrella (Opción 2)'],
    'Anomalía de Nombre (filas afectadas)':        ['Estrella (Opción 2)'],
}
PERDEDORES = {
    'Memoria RAM (KB)':                            ['Tabla Plana (Base)'],
    'Integridad Macro (Avg Growth %)':             ['Tabla Plana (Base)'],
    'Anomalía de Actualización (filas afectadas)': ['Tabla Plana (Base)'],
    'Anomalía de Nombre (filas afectadas)':        ['Tabla Plana (Base)', '3NF Inmon (Opción 1)'],
}

def resaltar_fila(row):
    metrica   = row['Métrica Analítica']
    ganadores = GANADORES.get(metrica, [])
    perdedores = PERDEDORES.get(metrica, [])
    return [
        VERDE if col in ganadores  else
        ROJO  if col in perdedores else ''
        for col in row.index
    ]

styled = (
    df_comparativo.style
    .apply(resaltar_fila, axis=1)
    .set_caption("Tabla Comparativa de Modelos de Datos — Entrega 3")
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding-bottom', '8px')]},
        {'selector': 'th',
         'props': [('background-color', '#2c3e50'), ('color', 'white'),
                   ('font-size', '11px'), ('text-align', 'center')]},
        {'selector': 'td',
         'props': [('font-size', '11px'), ('vertical-align', 'top'),
                   ('max-width', '210px'), ('white-space', 'normal')]},
    ])
    .set_properties(**{'text-align': 'center'})
    .set_properties(subset=['Conclusión de Evaluación'], **{'text-align': 'left'})
)

display(styled)

os.makedirs('../outputs', exist_ok=True)
df_comparativo.to_csv('../outputs/tabla_comparativa_modelos.csv', index=False)
print('Tabla comparativa exportada a /outputs/')

,Métrica Analítica,Tabla Plana (Base),3NF Inmon (Opción 1),Estrella (Opción 2),Conclusión de Evaluación
0,Memoria RAM (KB),3425.460000,586.380000,200.220000,Tabla Plana almacena todas las columnas descriptivas en cada fila transaccional. 3NF normaliza las dimensiones pero mantiene 'Partner Name' como FK string en el fact (7 783 strings repetidos). Estrella reemplaza ese string por int64 → mínima huella en la tabla de hechos.
1,Integridad Macro (Avg Growth %),1.89% (distorsionado),1.96% (real),1.96% (real),Tabla Plana repite World Growth % una vez por país/año: su promedio queda ponderado por el N de países activos cada año → fan-out trap. 3NF y Estrella aíslan el indicador en una tabla anual (Dim_Macro / Dim_Time): un único valor por año garantiza el promedio real.
2,Anomalía de Actualización (filas afectadas),238 filas (una por país activo en 2010),1 fila (solo en Dim_Macro_3NF),1 fila (solo en Dim_Time),3NF y Estrella aíslan World Growth en tablas dedicadas: corregir un año requiere 1 fila. En Tabla Plana la corrección se propaga a todas las filas del año (una por país activo en 2010).
3,Anomalía de Nombre (filas afectadas),34 filas (en dataset completo),35 filas (fact + dim; FK es string),1 fila (Dim_Country; fact intacto con SK int),"En Tabla Plana y 3NF el 'Partner Name' es texto en el fact: un cambio de nombre oficial exige actualizar cada fila que lo referencie. Paradójicamente, 3NF resulta levemente peor que Tabla Plana al requerir las mismas filas del fact más 1 fila adicional en Dim_Partners_3NF. Estrella es inmune: el fact solo almacena el int64 surrogate; basta actualizar 1 fila en Dim_Country."


Tabla comparativa exportada a /outputs/


## 5. Exportación del Modelo Ganador (Esquema en Estrella)
El benchmark evidencia que el **Esquema en Estrella** obtiene el mejor resultado en las 4 métricas evaluadas frente a Tabla Plana y 3NF Inmon. Se exportan las tres tablas que lo componen como fuente de datos para Tableau.

In [18]:
os.makedirs('../outputs/tableau_sources', exist_ok=True)
fact_trade.to_csv('../outputs/tableau_sources/Fact_Trade.csv', index=False)
dim_country.to_csv('../outputs/tableau_sources/Dim_Country.csv', index=False)
dim_time.to_csv('../outputs/tableau_sources/Dim_Time.csv', index=False)

print('Fuentes exportadas para Tableau:')
print(f'  Fact_Trade  → {fact_trade.shape[0]} filas, {fact_trade.shape[1]} cols')
print(f'  Dim_Country → {dim_country.shape[0]} filas, {dim_country.shape[1]} cols')
print(f'  Dim_Time    → {dim_time.shape[0]} filas, {dim_time.shape[1]} cols')

Fuentes exportadas para Tableau:
  Fact_Trade  → 7783 filas, 3 cols
  Dim_Country → 252 filas, 2 cols
  Dim_Time    → 34 filas, 3 cols


## 6. Reporte Ejecutivo de Decisión

### Modelo seleccionado: Esquema en Estrella (Star Schema — Kimball)

La decisión se basa exclusivamente en evidencia empírica del benchmark (Sección 3).

| Criterio | Tabla Plana (Base) | 3NF Inmon (Opción 1) | Estrella (Opción 2) | Veredicto |
|:---|:---:|:---:|:---:|:---|
| Memoria RAM | ~3 432 KB | ~660 KB | ~200 KB | Estrella minimiza la huella |
| Integridad `World Growth (%)` | distorsionado (fan-out) | ✓ real | ✓ real | **Descarta Tabla Plana** |
| Anomalía de actualización | N filas/año | 1 fila en Dim_Macro | 1 fila en Dim_Time | 3NF y Estrella empatan |
| Anomalía de nombre | N filas | N+1 filas (fact + dim) | 1 fila en Dim_Country | **Descarta 3NF** |

**Razonamiento:**

El **Modelo 3NF (Inmon)** mejora significativamente sobre la Tabla Plana: elimina el fan-out trap (métrica 2) y la anomalía de actualización de World Growth (métrica 3). Sin embargo, presenta una debilidad estructural crítica que lo descarta: al mantener `Partner Name` (string) como clave foránea en la tabla de hechos, cualquier cambio en el nombre oficial de un país exige actualizar **cada fila del fact** que referencie ese país. Paradójicamente, el costo es mayor que en la Tabla Plana, ya que requiere las mismas N filas del fact más 1 fila adicional en `Dim_Partners_3NF`.

El **Esquema en Estrella** resuelve este problema con las **surrogate keys enteras**: `Fact_Trade` solo almacena `dim_country_sk` (int64). Un cambio de nombre en `Dim_Country` no toca el fact — es inmune estructuralmente. Adicionalmente, los joins sobre `int64` son más eficientes que sobre `varchar`, ventaja crítica en Tableau con datasets de decenas de miles de filas.

```
Fact_Trade ──► Dim_Country   (252 países — surrogate key int64)
           ──► Dim_Time      (34 años + World Growth % — surrogate key int64)
```

---

> El reporte completo está en `docs/entrega3-reporte-modelado.md`.  
> Las tablas del modelo ganador están en `outputs/tableau_sources/` listas para Tableau.